# 05 · 工程化与总结

> Multi-Agent Frameworks：协作优化 · 监督与可观测性 · 最佳实践 · 常见陷阱

本节是 Multi-Agent 模块的收尾。我们把前面学到的 AutoGen、CrewAI、LangGraph / Agentic RAG
放回工程视角，回答一个关键问题：**如何让多 Agent 系统稳定上线？**

## 学习目标

1. 理解 Agent 间通信与协作的工程优化手段
2. 建立由 Trace、指标、质量门与 HITL 组成的监督体系
3. 能根据问题结构选择框架与协作模式
4. 识别并规避多 Agent 系统的常见工程陷阱

## 目录

1. 协作与通信优化
2. 监督与可观测性
3. Agentic RAG 总体架构
4. 三种典型协作模式
5. 最佳实践
6. 常见陷阱
7. 框架选型与接入速查
8. 课程小结与延伸阅读

## 1. 协作与通信优化

> 多 Agent 的瓶颈常常不在模型本身，而在**通信、调度和状态管理**。
> 因此要像设计分布式系统一样设计协作，不同框架只是入口不同，工程约束并不会消失。

### 1.1 同步 vs 异步

| 通信方式 | 适合场景 | 说明 |
|----------|----------|------|
| 同步 RPC | 必须立刻返回的步骤 | 关键路径短小、实时决策 |
| 异步消息队列 | 需要缓冲和扩展的步骤 | 大吞吐、不稳定步骤 |
| 显式状态推进 | 分支多、需恢复的流程 | 状态可持久化、可重放 |

**混合架构最常见**：关键路径用同步，大吞吐和不稳定步骤用异步，不要期待一种方式通吃。

### 1.2 Schema：消息、任务与状态

给协作载体定义固定结构，让不同 Agent “说同一种语言”。它可以是 message schema、task contract 或 state schema。

推荐字段：

- `job_id`：一次任务/作业的标识
- `task_id`：作业内子任务标识
- `schema_version`：支持结构逐步演进
- `payload`：任务输入
- `retry_count` / `status`：重试与状态追踪

> Schema 是重试、回放和审计的前提。

### 1.3 幂等与重试

- 同一个任务重发时不应产生重复副作用
- 用 `job_id + task_id` 作为幂等键
- 重试前先判断是临时错误还是逻辑错误
- 把重试设计进系统，而不是事故后补救

常见策略：**立即重试 → 指数退避 → 最大次数上限**。

### 1.4 优先级与反压（Backpressure）

- `realtime` 队列不要被 `batch` 任务堵住
- 高优先级任务使用独立通道
- 反压可以防止系统把自己压垮
- 预算与速率限制也属于反压的一部分

没有反压时：队列越堆越长、延迟越来越大、成本失控。

### 1.5 信息压缩

Agent 之间传**摘要**而不是整篇原文，传**引用**而不是整份上下文：

```text
文档：doc#page#chunk  →  只传下一步真正需要的 chunk
```

收益：降低 token、降低延迟、减少上下文污染。

> 这与“选一个大上下文模型”不是一回事。

## 2. 监督与可观测性

### 2.1 Trace 与回放

Trace 记录每一步的输入和输出，是多 Agent 系统最关键的调试材料。要能从 Trace 中看出“卡在哪一步”。

推荐记录：

- prompt
- retrieved context
- tool calls
- model output
- final decision

Trace 的三种用途：**回放复现问题、审计决策链路、定位失败步骤**。

### 2.2 监控指标

| 指标 | 说明 |
|------|------|
| latency | 端到端与各步骤耗时，关注 p95/p99 |
| error rate | 失败率、工具调用失败率 |
| queue length | 队列积压，反映系统压力 |
| task success rate | 任务完成率 |
| cost / token | 单次调用与整体成本 |
| verifier pass rate | 质量门通过率 |

> 系统变差时，先看指标，不要先怪模型。

### 2.3 质量门（Quality Gates）

在关键阶段（尤其是最终答案生成前）插入 verifier Agent：

- 检索复核
- NLI（自然语言推理）检验
- 规则检查

质量门应当是流程的一部分，而不是补丁；它让系统拥有**可控失败**的出口。

### 2.4 人工复核（HITL）

当自动验证不确定时，交给人处理。HITL 不是倒退，而是质量策略的一部分。

适合 HITL 的场景：

- 高风险内容
- 低置信度结果
- 涉及合规或业务决策

人类介入点越清楚，系统越可治理。人工结果还可以反哺后续优化。

### 2.5 安全与治理

- Agent 只能访问必要资源（最小权限）
- 危险能力必须白名单 + 审计
- 考虑隐私、数据保留与合规
- 工具权限要最小化，不要默认全开
- 数据流向要可解释、可追踪、可回收

不要把以下能力随便开放给 Agent：**shell、数据库写权限、高权限外部 API 密钥**。

## 3. Agentic RAG 总体架构

```text
用户问题
   │
   ▼
Retriever    找候选资料
   │
   ▼
Reranker     精排候选
   │
   ▼
Generator    生成答案
   │
   ▼
Verifier     检查关键断言
   │
   ▼
Orchestrator 决定放行 / 重试 / 转人工
```

| 角色 | 职责 |
|------|------|
| Retriever | 从知识库召回候选文档 |
| Reranker | 对候选做精排，缩小上下文 |
| Generator | 基于上下文生成答案 |
| Verifier | 核查答案中的关键断言是否有依据 |
| Orchestrator | 编排流程，决定放行、重试或转人工 |

这就是最常见的多 Agent RAG 骨架，也是“协作 + 监督”最适合的教学案例。

## 4. 三种典型协作模式

### 示例 A：AutoGen 风格三角色流水线

角色：`Researcher → Writer → Editor`，适合快速展示“角色分工”。

```python
researcher = Agent(name="Researcher", capability="search_and_extract")
writer = Agent(name="Writer", capability="compose_summary")
editor = Agent(name="Editor", capability="fact_check_and_edit")

def orchestrate(topic):
    docs = researcher.run({"task": "search", "query": topic})
    draft = writer.run({"task": "write", "context": docs})
    final = editor.run({"task": "verify_and_edit", "draft": draft, "context": docs})
    return final
```

**教学点**

- Researcher 只负责找资料
- Writer 只负责组织表达
- Editor 只负责核查和改写
- 每个 Agent 的边界要清楚

> 课堂提问：如果 Writer 直接去检索，会发生什么？

### 示例 B：CrewAI / 队列风格编队

Orchestrator 把任务切成子任务，Worker 通过队列逐步处理，适合高并发、长流程和可扩容任务链。

```python
job_id = create_job(topic)
push_queue("retriever", {"job_id": job_id, "topic": topic})

# retriever worker
# reranker worker
# generator worker
# result db
```

**工程点**

- 每个队列都能水平扩容 Worker
- `job_id` 用于结果聚合
- 队列长度能直接反映系统压力
- Worker 健康检查决定是否自动重试

### 示例 C：Agentic RAG 多 Agent 协作

Retriever、Reranker、Generator、Verifier 分工明确，Orchestrator 负责最终放行。

```python
retrieval_job = dispatch("retriever", {"query": q, "top_k": 50})
candidates = wait_for_result(retrieval_job)
top5 = call_agent("reranker", {"query": q, "candidates": candidates})
draft = call_agent("generator", {"query": q, "context": top5})
verify = call_agent("verifier", {"draft": draft, "context": top5})

if verify["pass"]:
    return draft
else:
    send_to_human_review(draft, verify)
```

### 三种模式的建模对照

| 协作模式 | AutoGen | CrewAI | LangGraph |
|----------|---------|--------|-----------|
| A 角色流水线 | 最自然（GroupChat / GraphFlow） | 改写为 3 个 Task | 3 个顺序节点 + 共享 state |
| B 队列编队 | 适合做前置原型 | 最贴近任务流 | 队列内关键决策的子流程 |
| C Agentic RAG | 适合演示协作与审校 | 适合任务链 | 最适合表达 verify 分支 |

> 先选抽象，再选框架：对话问题用对话抽象，任务问题用任务抽象，状态问题用状态抽象。

## 5. 最佳实践

最佳实践不是“写更多代码”，而是让系统**更稳、更可解释、更可扩展**。

### 5.1 职责分解与抽象匹配

- 单一职责优先，一个 Agent 不要做太多事
- 职责过载会导致：调试变难、重试变危险、扩展变贵
- 框架抽象要与问题结构匹配

### 5.2 Schema 与版本化

- 消息 schema、task contract、state schema 都要版本化
- 兼容旧结构，允许逐步演进
- 推荐字段：`schema_version`、`job_id`、`task_id`、`payload`、`trace_id`

### 5.3 幂等、Trace 与审计

- 幂等 key 必须设计好
- 失败重试必须可控
- Trace 要能回放
- 关键步骤要可审计

> 这四件事一起决定系统能不能进入生产。

### 5.4 先模拟异常，再接真实世界

测试顺序很重要：**正常路径 → 失败路径 → 边界路径 → 恢复路径**。

1. 在沙箱里模拟异常
2. 验证降级与恢复是否有效
3. 接入真实数据与真实工具
4. 再考虑规模化扩容

### 5.5 预算、成本与长期演进

- 预算和成本要前置考虑
- 高成本 Agent 要有降级方案
- 先用小模型预筛，再用大模型精处理
- 框架要保留迁移路径，避免被锁死

> 原则：先证明价值，再扩大规模。

## 6. 常见陷阱

> 多 Agent 系统的问题，往往不是“不会调用 API”，而是“没有把失败设计进去”。
> 另一个隐藏问题是：选错抽象层，会把简单问题做复杂。

| # | 陷阱 | 表现 | 解决方式 |
|---|------|------|----------|
| 1 | 没有停止条件的循环 | Agent 互相触发，成本和延迟失控 | 步数上限、时间上限、成本上限 |
| 2 | 过度中心化的 Orchestrator | 瓶颈、单点故障、演进困难 | 分片、队列化、事件驱动 |
| 3 | 高基数日志导致成本暴涨 | 全量 trace 又贵又慢 | 常规采样、异常全量、关键链路保留 |
| 4 | 共享可变状态导致竞态 | 并发写入互相覆盖 | 事务、乐观锁、明确写入权责 |
| 5 | 直接暴露危险能力 | 无审计、无白名单，影响被放大 | 权限隔离、操作审计、人工兜底 |
| 6 | 忽视隐私、合规与选型边界 | 功能跑通但生产不能上；为炫技而上复杂框架 | 先确认数据能否传、结果能否留、动作能否自动执行 |

**生产前先问自己：**

1. 这份数据能不能传？
2. 这个结果能不能留？
3. 这个动作能不能自动执行？

## 7. 框架选型与接入速查

### 7.1 核心抽象对照

| 框架 / 模式 | 核心抽象 | 适用场景 |
|-------------|----------|----------|
| AutoGen GraphFlow | `DiGraphBuilder` + `GraphFlow` | 固定顺序流水线 |
| AutoGen SelectorGroupChat | `selector_prompt` | 灵活多轮讨论 |
| AutoGen Memory | `ListMemory` | 跨轮次上下文 |
| CrewAI Sequential | `Agent + Task + Crew` | 高层级角色协作 |
| CrewAI Hierarchical | `Process.hierarchical` | 动态任务分配 |
| Agentic RAG | LangGraph `StateGraph` | 检索 → 生成 → 验证 |
| Supervisor + HITL | `pass_rate` 阈值 | 生产级质量保障 |

### 7.2 OpenRouter 接入方式

| 框架 | 接入方式 |
|------|----------|
| AutoGen | `OpenAIChatCompletionClient(base_url=OPENROUTER_BASE_URL)` |
| CrewAI | `LLM(model='openrouter/...', base_url=OPENROUTER_BASE_URL)` |
| LangChain | `ChatOpenAI(openai_api_base=OPENROUTER_BASE_URL)` |

### 7.3 成本分层示例

| 任务类型 | 推荐模型档位 | 原因 |
|----------|--------------|------|
| 短摘要 / 抽取 | 小模型（如 Haiku） | 便宜、足够用 |
| 主力生成 / 风险抽取 | 中档模型（如 Sonnet） | 质量与成本平衡 |
| 复杂推理 | 大模型（如 Opus） | 按需使用，控制频率 |

### 7.4 五条设计原则

1. **单一职责**：每个 Agent 只做一件事
2. **清晰的 State / Message Schema**：便于调试和重放
3. **终止条件**：始终设置最大步数 / 成本上限
4. **质量门**：关键节点插入 Verifier
5. **成本感知**：复杂任务用大模型，轻量任务用小模型

> Supervisor / HITL 的完整可运行示例见 `05-agent_ops_langsmith_demo.ipynb`。

## 8. 课程小结与延伸阅读

### 8.1 小结

Multi-Agent Frameworks（AutoGen、CrewAI、LangGraph）为复杂任务提供了模块化、可扩展、易维护的实现路径。

- **关键能力**：职责拆分、消息/事件协作、幂等与重试、Trace 与监督、质量门与 HITL
- **实现路径**：先用 AutoGen 风格原型验证角色与流程，再用 CrewAI / 编队 + 消息队列做生产级扩展
- **工程前提**：安全、可观测性、成本控制与可回放

### 8.2 一页记住

- 先决定职责，再决定框架
- 消息、任务、状态是三种不同的协作载体
- Trace、Quality Gates、HITL 是生产前提
- 不要为了多 Agent 而多 Agent
- 先证明价值，再扩大规模

### 8.3 术语速查

| 术语 | 含义 |
|------|------|
| Agent | 执行单一职责的智能体 |
| Orchestrator | 负责编排流程的调度者 |
| Verifier | 负责质量检查的 Agent |
| HITL | Human-in-the-loop，人工复核 |
| Trace | 系统执行轨迹 |
| Checkpoint | 流程可暂停、可恢复的中间状态 |

### 8.4 延伸阅读

- [AutoGen 官方文档](https://microsoft.github.io/autogen/stable/)
- [CrewAI 官方文档](https://docs.crewai.com/)
- [LangGraph 官方文档](https://docs.langchain.com/langgraph)
- [LangGraph 持久化与恢复](https://docs.langchain.com/oss/python/langgraph/persistence)
- 多 Agent 评估与 RAGAS 相关资料

> 建议先从一个“小流程”开始，而不是上来就做大而全的平台。

### Q & A

回想一下：今天最想落地的一个多 Agent 场景是什么？